In [ ]:
# 1️⃣ استيراد الحزم اللازمة
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
import pandas as pd

In [ ]:

# 2️⃣ تحميل ملف CSV
books = pd.read_csv("books_cleaned.csv")

# التأكد أن العمود موجود
if "tagged_description" not in books.columns:
    raise ValueError("Column 'tagged_description' does not exist in the DataFrame.")

In [ ]:

# 3️⃣ حفظ كل وصف في سطر منفصل في ملف txt
with open("tagged_description.txt", "w", encoding="utf-8") as f:
    for desc in books["tagged_description"].astype(str):
        f.write(desc.strip() + "\n")

print("File 'tagged_description.txt' created successfully!")

In [ ]:

# 4️⃣ تحميل النصوص من الملف
raw_documents = TextLoader("tagged_description.txt", encoding="utf-8").load()

In [ ]:

# 5️⃣ تقسيم النصوص إلى chunks
text_splitter = CharacterTextSplitter(
    chunk_size=500,    # حجم كل جزء
    chunk_overlap=50,  # التداخل بين الأجزاء
    separator="\n"
)
documents = text_splitter.split_documents(raw_documents)
print(f"Loaded {len(documents)} documents successfully!")

In [ ]:

# 6️⃣ إنشاء embeddings باستخدام HuggingFace
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:

# 7️⃣ إنشاء قاعدة بيانات Chroma من النصوص
db_books = Chroma.from_documents(
    documents,
    embedding=embedding_model,
    persist_directory="chroma_books"  # تحفظ قاعدة البيانات على الجهاز
)
print("Chroma database created successfully!")

In [ ]:

# 8️⃣ مثال على البحث
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k=10)
print("Top document content:\n", docs[0].page_content)

In [ ]:

# 9️⃣ استرجاع البيانات من CSV بناءً على النتائج
isbn_match = int(docs[0].page_content.split()[0].strip())
print(books[books["isbn13"] == isbn_match])

In [65]:

# 10️⃣ دالة لاسترجاع التوصيات بشكل منظم
def retrieve_semantic_recommendations(query: str, top_k: int = 10) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=top_k)
    books_list = [int(rec.page_content.strip('"').split()[0]) for rec in recs]
    return books[books["isbn13"].isin(books_list)]

# مثال على استخدام الدالة
recommendations = retrieve_semantic_recommendations("A book to teach children about nature")
print(recommendations)


Created a chunk of size 1168, which is longer than the specified 500
Created a chunk of size 1214, which is longer than the specified 500
Created a chunk of size 960, which is longer than the specified 500
Created a chunk of size 843, which is longer than the specified 500
Created a chunk of size 877, which is longer than the specified 500
Created a chunk of size 1088, which is longer than the specified 500
Created a chunk of size 1189, which is longer than the specified 500
Created a chunk of size 513, which is longer than the specified 500
Created a chunk of size 752, which is longer than the specified 500
Created a chunk of size 728, which is longer than the specified 500
Created a chunk of size 721, which is longer than the specified 500
Created a chunk of size 1253, which is longer than the specified 500
Created a chunk of size 681, which is longer than the specified 500
Created a chunk of size 553, which is longer than the specified 500
Created a chunk of size 521, which is longe

File 'tagged_description.txt' created successfully!


Created a chunk of size 682, which is longer than the specified 500
Created a chunk of size 598, which is longer than the specified 500
Created a chunk of size 663, which is longer than the specified 500
Created a chunk of size 595, which is longer than the specified 500
Created a chunk of size 851, which is longer than the specified 500
Created a chunk of size 595, which is longer than the specified 500
Created a chunk of size 537, which is longer than the specified 500
Created a chunk of size 563, which is longer than the specified 500
Created a chunk of size 705, which is longer than the specified 500
Created a chunk of size 848, which is longer than the specified 500
Created a chunk of size 542, which is longer than the specified 500
Created a chunk of size 1177, which is longer than the specified 500
Created a chunk of size 904, which is longer than the specified 500
Created a chunk of size 852, which is longer than the specified 500
Created a chunk of size 507, which is longer th

Loaded 4414 documents successfully!
Chroma database created successfully!
Top document content:
 9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.
             isbn13      isbn10                                title  \
3747  9780786808069  0786808063  Baby Einstein: Neighborhood Animals   

                                authors        categories  \
3747  Marilyn Singer;Julie Aigner-Clark  Juvenile Fiction   

                                              thumbnail  \
3747  http://books.google.com/books/content?id=X9a4P...   

                                            description  published_year  \
3747  Children will discover the exciting world of t...          2001.0   

      average_rating  num_pages  ratings_count  \
3747            3.89       16.0      